# **PROJECT 7:** AI-Powered Plant Disease Detection with Generative AI Explanations

### **Step-1: Upload the saved model zip file (downloaded from Teachable Machine) in Google Colab**

In [1]:
#@title Upload the saved model zip file and Run this cell to Unzip it

import os
from IPython.display import clear_output

# Get all .zip files in the current folder
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if not zip_files:
    print("Please upload the saved model zip file (downloaded from Teachable Machine) in Google Colab.")
else:
    !unzip "$(ls | grep '.zip')"
    clear_output()
    print("Model Unzipped successfully!")

Model Unzipped successfully!


### **Step-2: Load Model and Class Labels**

In [2]:
#@title Run this cell to Import required libraries

import tensorflow as tf
from PIL import Image, ImageOps
import numpy as np
# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
#@title Run this cell to Load the SavedModel

# ----------------------------
# Load SavedModel
# ----------------------------
saved_model = tf.saved_model.load("model.savedmodel")

# Get the serving function
infer = saved_model.signatures["serving_default"]

print("Model loaded successfully!")

Model loaded successfully!


In [3]:
#@title Run this cell to Load Class labels

class_names = open("labels.txt", "r").readlines()
class_names = [name.split()[-1] for name in class_names]

print("Class labels loaded successfully!")
print(class_names)

Class labels loaded successfully!
['Apple_Scab', 'Apple_Healthy', 'Potato_Early_Blight', 'Potato_Healthy', 'Tomato_Leaf_Mold', 'Tomato_Healthy']


### **Step-3: Upload a Test image and Get Prediction**

In [4]:
#@title Upload a Plant Leaf Test image, provide its file name, then Run this cell

# "Upload a test image in Colab and provide its file name here (eg. Apple_scab_test_1.JPG)"
img_file = 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_2.JPG'  #@param {type:"string"}

# Prepare the input image
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

image = Image.open(img_file).convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

# Show image
image_array

array([[[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [177, 171, 207],
        [177, 171, 207],
        [177, 171, 207]],

       [[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [176, 170, 206],
        [176, 170, 206],
        [176, 170, 206]],

       [[165, 153, 189],
        [165, 153, 189],
        [165, 153, 189],
        ...,
        [174, 168, 204],
        [174, 168, 204],
        [174, 168, 204]],

       ...,

       [[144, 128, 165],
        [137, 121, 158],
        [135, 119, 156],
        ...,
        [138, 128, 163],
        [143, 133, 168],
        [147, 137, 172]],

       [[137, 121, 158],
        [131, 115, 152],
        [130, 114, 151],
        ...,
        [141, 131, 166],
        [145, 135, 170],
        [142, 132, 167]],

       [[138, 122, 159],
        [133, 117, 154],
        [136, 120, 157],
        ...,
        [149, 139, 174],
        [148, 138, 173],
        [140, 130, 165]]

In [13]:
import glob
images = glob.glob("Plant_leave_diseases_dataset/**/*.[jJ][pP]*[gG]", recursive=True)
print(len(images))
print(images[:5])

204
['Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_1.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_2.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_3.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_healthy_test_4.JPG', 'Plant_leave_diseases_dataset\\Testing\\Apple_scab_test_1.JPG']


In [14]:
#@title Run this cell to Get Prediction for the test image

# ----------------------------
# Run Prediction
# ----------------------------
# Convert numpy → tensor
input_tensor = tf.convert_to_tensor(data)

# Call the SavedModel signature
output = infer(input_tensor)

# SavedModel may return a dict – get the first output tensor
prediction = list(output.values())[0].numpy()

# Decode prediction
index = np.argmax(prediction)
class_name = class_names[index].strip()
confidence_score = prediction[0][index]

# Print results
print("Class:", class_name)
print("Confidence Score:", confidence_score)


NameError: name 'infer' is not defined

### **Step-4: Provide the Google Gemini API Key**

* Use an existing Google Gemini API key, or Create a new one by:
    * Visiting https://ai.google.dev/gemini-api/docs, and
    * Clicking on *Get a Geminin API key*
    * Note that Google Gemini's Free plan includes limited access to certain models



In [15]:
#@title Enter your Gemini API Key
import os

GEMINI_API_KEY = input("Enter your Gemini API Key: ")
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY



### **Step-5: Define prompt to feed to an LLM to get farmer-friendly explanations, treatment recommendations, and optional multilingual support for regional accessibility.**

Adjust the prompt as required.

In [16]:
# ----------------------------
# Define Prompt
# ----------------------------

language = "English"

# Prompt For Response Generation
# Use a fallback so this cell still works even if the prediction cell hasn't run yet
predicted_disease = globals().get("class_name", "Unknown disease")

prompt = f"""
You are an expert Agricultural plant pathologist and crop-health advisor.
Explain the condition {predicted_disease} and provide practical, safe, and easy-to-apply remedies.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions a farmer/gardener should take
- Natural or low-cost remedies
- When to consider chemical treatments
- Preventive measures for future crops

Symptom: {predicted_disease}

Answer only in {language}.

If the plant is unhealthy, then start the response in this format:
`**Plant:** <plant-name>
 **Status:** Unhealthy
 **Disease:** <disease-name>`

If the plant is healthy, then respond in this format:
`**Plant:** <plant-name>
 **Status:** Healthy`
"""

print("\nPROMPT\n" + "="*100)
print(prompt)


PROMPT

You are an expert Agricultural plant pathologist and crop-health advisor.
Explain the condition Unknown disease and provide practical, safe, and easy-to-apply remedies.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions a farmer/gardener should take
- Natural or low-cost remedies
- When to consider chemical treatments
- Preventive measures for future crops

Symptom: Unknown disease

Answer only in English.

If the plant is unhealthy, then start the response in this format:
`**Plant:** <plant-name>
 **Status:** Unhealthy
 **Disease:** <disease-name>`

If the plant is healthy, then respond in this format:
`**Plant:** <plant-name>
 **Status:** Healthy`



### **Step-6: Make an LLM API call using the above prompt**

In [17]:
#@title Run this cell to Make an LLM API Call

# ----------------------------
# Make an LLM Call
# ----------------------------

# Initialize Client
!pip install -q google-genai
import os
# Install the SDK in the current notebook environment
!pip install -q google-genai

# Import the SDK
# Install the SDK in the current notebook environment
%pip install -q google-genai

from google import genai

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
gemini_client = genai.Client()


# Make API call and get response
gemini_response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

# Show response in Markdown
from IPython.display import Markdown

Markdown(gemini_response.text)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Plant:** Affected Plant/Crop
**Status:** Unhealthy
**Disease:** Unknown disease

As an expert Agricultural Plant Pathologist and Crop-Health Advisor, I understand the frustration and concern when an "unknown disease" strikes your plants. This term doesn't refer to a specific pathogen but rather a situation where a plant is exhibiting symptoms of distress, but the underlying cause has not yet been identified. It's a diagnostic challenge that requires careful observation and a systematic approach.

### Brief Explanation of the Condition (Unknown Disease)

"Unknown disease" is the initial assessment when a plant displays symptoms of ill health (e.g., wilting, yellowing, spots, stunted growth) but the specific pest, pathogen (fungus, bacteria, virus, nematode), environmental stress, or nutrient deficiency causing these symptoms is not immediately apparent. It's a call to action for further investigation, as an accurate diagnosis is crucial for effective treatment.

### Key Symptoms

Since the cause is unknown, the symptoms can be varied and non-specific. You might observe:

*   **General Decline:** Wilting, drooping, stunted growth, reduced vigor, abnormal leaf drop.
*   **Leaf Discoloration:** Yellowing (chlorosis), browning (necrosis), purpling, red tints, mottling, or unusual spots and lesions that don't fit known patterns.
*   **Abnormal Growths:** Galls, cankers, swellings, or distorted leaves (curling, twisting), stems, or fruits.
*   **Root Problems:** (If inspectable) Discolored, softened, rotted, or abnormally stunted roots.
*   **Exudates:** Unexplained oozing of sap or sticky substances from stems or fruits.
*   **Unusual Patterns:** Symptoms appearing in patches, affecting only specific parts of the plant, or spreading rapidly without an obvious cause.
*   **Lack of Visible Pests:** No easily identifiable insects, mites, or other larger pests.

### Immediate Actions a Farmer/Gardener Should Take

1.  **Isolate & Observe:** If possible, quarantine or remove severely affected plants to prevent potential spread, especially if a contagious pathogen is suspected. Immediately begin detailed observation.
2.  **Document Everything:** Take clear photos of the symptoms (close-up and whole plant), note the date of onset, progression, pattern of spread, and environmental conditions (temperature, humidity, recent rainfall, irrigation schedule, recent fertilizer/pesticide applications). This documentation is vital for diagnosis.
3.  **Check Environmental Factors:** Rule out obvious non-infectious causes first:
    *   **Water:** Is the plant getting too much or too little water? Check soil moisture.
    *   **Nutrients:** Has there been recent fertilization? Could it be a deficiency or toxicity?
    *   **Temperature:** Has there been a recent heatwave, cold snap, or frost?
    *   **Light:** Is the plant getting adequate or excessive sunlight for its needs?
    *   **Soil:** Is the soil compacted? Does it drain well? What's the pH?
    *   **Chemical Exposure:** Could there have been herbicide drift or accidental chemical application?
4.  **Thorough Pest Inspection:** Even if no obvious pests are seen, use a hand lens to carefully inspect the undersides of leaves, stems, and growing points for tiny insects (mites, aphids, thrips, whiteflies), their eggs, or webbing.
5.  **Stop Suspect Practices:** If you recently applied a new product (fertilizer, pesticide, soil amendment), temporarily cease its use on a small test area.
6.  **Collect Samples (for potential diagnosis):** If symptoms are severe or spreading, collect samples (leaves, stems, roots showing transitional symptoms – i.e., both healthy and affected tissue) for potential professional diagnosis. Place samples in a dry plastic bag and store in a cool place, avoiding direct sunlight.

### Natural or Low-Cost Remedies (Supportive Care)

While a specific diagnosis is pending, focus on general plant health and supportive care:

1.  **Optimize Watering:** Ensure consistent, appropriate watering. Avoid both waterlogging (which can lead to root rot) and drought stress. Water deeply and less frequently, allowing the topsoil to dry slightly between waterings.
2.  **Improve Soil Health:**
    *   **Organic Matter:** Incorporate well-rotted compost or other organic matter into the soil. This improves soil structure, drainage, water retention, and nutrient availability, fostering a healthier root environment and beneficial microbial activity.
    *   **Mulching:** Apply a layer of organic mulch (e.g., straw, wood chips) around plants. This helps regulate soil temperature, conserves moisture, suppresses weeds, and slowly releases nutrients.
3.  **Gentle Nutrient Support:**
    *   **Compost Tea/Seaweed Extract:** Apply a mild compost tea or seaweed extract as a foliar spray or soil drench. These can provide trace elements and growth hormones that help plants recover from stress and boost natural immunity, without causing nutrient imbalances.
    *   **Avoid heavy synthetic fertilizers:** Until the problem is known, avoid applying strong synthetic fertilizers, as they can sometimes exacerbate certain issues or mask symptoms.
4.  **Pruning:** Carefully prune away severely affected plant parts (if localized) using sterilized tools (wipe with rubbing alcohol or a 10% bleach solution between cuts). Dispose of diseased material away from healthy plants; do not compost it.
5.  **Introduce Beneficial Microbes:** Consider applying products containing beneficial soil microbes like mycorrhizal fungi or *Bacillus subtilis*. These can enhance nutrient uptake, improve root health, and suppress some soil-borne pathogens.

### When to Consider Chemical Treatments

**Chemical treatments for an "unknown disease" are generally NOT recommended and can be counterproductive.** Applying broad-spectrum chemicals without knowing the specific problem is often:

*   **Ineffective:** You might be treating for the wrong issue, wasting time and money.
*   **Harmful:** It can damage the plant, beneficial insects, soil microbes, and the environment.
*   **Masking:** It can obscure symptoms, making a proper diagnosis even harder.

**Consider chemical treatments ONLY under these specific circumstances:**

*   **After a Confirmed Diagnosis:** Once a professional diagnosis identifies a specific pathogen or pest, then a targeted chemical (e.g., specific fungicide, bactericide, or insecticide) can be considered as a last resort, strictly following label instructions.
*   **Rapid, Aggressive Spread with High Economic Impact:** In commercial settings, if an "unknown" problem is spreading exceptionally fast, threatening the entire crop, and causing significant economic losses, a very cautious application of a broad-spectrum, low-toxicity product (e.g., neem oil for suspected pests, or a copper-based product for suspected bacterial/fungal issues) might be considered on a small test area, *while still actively pursuing a definitive diagnosis*. This is a desperate measure and should be done with extreme caution.
*   **Professional Recommendation:** Always consult with a local agricultural extension office or a plant pathology lab before applying any broad-spectrum chemical to an unknown problem.

### Preventive Measures for Future Crops

Proactive measures are the best defense against *any* disease, known or unknown:

1.  **Crop Rotation:** Practice proper crop rotation by not planting the same type of crop (or closely related crops) in the same spot year after year. This disrupts disease cycles and prevents the buildup of soil-borne pathogens.
2.  **Sanitation:**
    *   **Clean Tools:** Regularly clean and sanitize gardening tools, pots, and equipment with a 10% bleach solution or rubbing alcohol.
    *   **Remove Debris:** Promptly remove and dispose of all plant debris, fallen leaves, and diseased material from the garden area to eliminate pathogen reservoirs.
3.  **Choose Resistant Varieties:** Whenever possible, select plant varieties that are known to be resistant or tolerant to common diseases in your region.
4.  **Healthy Soil Management:**
    *   **Soil Testing:** Conduct regular soil tests to monitor nutrient levels and pH. Amend the soil as needed to maintain optimal conditions for plant growth.
    *   **Organic Matter:** Continuously enrich your soil with compost and other organic matter to promote robust soil biology and improve overall soil health.
    *   **Drainage:** Ensure good soil drainage to prevent waterlogging, which can lead to root diseases.
5.  **Proper Spacing and Air Circulation:** Provide adequate space between plants to allow for good air circulation, which reduces humidity and discourages fungal and bacterial diseases.
6.  **Optimal Watering Practices:** Water at the base of plants in the morning, rather than overhead, to allow foliage to dry quickly, minimizing conditions favorable for disease development.
7.  **Quarantine New Plants:** Before introducing new plants to your garden, isolate them for 2-4 weeks to observe for any signs of pests or diseases.
8.  **Regular Monitoring:** Make it a habit to inspect your plants frequently (daily or weekly) for any unusual changes. Early detection is key to preventing problems from becoming severe and "unknown."
9.  **Understand Your Plants and Environment:** Learn the specific needs of the plants you grow and common problems (pests and diseases) prevalent in your local climate. This knowledge helps in anticipating and preventing issues.

By following these guidelines, you can significantly reduce the incidence of "unknown diseases" and build a more resilient and healthy garden or farm.

### **Step-7: Create a Gradio Interface**

Use the Gemini present within Google Colab notebook ( <img src='https://drive.google.com/uc?id=19N2vjh7eihxQ3qj9uavYblyveidsLKsf' width=30> ), to generate code for creating a Gradio Interface with below features.



Inputs:
- `Gradio File` to let user upload an image file
- `Gradio Textbox` to let user enter his preferred Language

Outputs:
- `Gradio Image` to show the image uploaded by user
- `Gradio Markdown` to render the LLM response

Logic:
- The user input image should go to the plant disease detection model (trained on Teachable Machine)
- The predicted disease-label and the user preferred language should become part of the prompt
- Use an LLM to generate farmer-friendly explanations, treatment guidance, and preventive steps


In [23]:
import tensorflow as tf
saved_model = tf.saved_model.load("model.savedmodel")
infer = saved_model.signatures["serving_default"]
with open("labels.txt") as f:
    class_names = [line.split()[-1] for line in f if line.strip()]
print("Loaded:", class_names)

Loaded: ['Apple_Scab', 'Apple_Healthy', 'Potato_Early_Blight', 'Potato_Healthy', 'Tomato_Leaf_Mold', 'Tomato_Healthy']


In [ ]:
# ============================================================
# AI PLANT DISEASE DETECTION + GEMINI EXPLANATION + GRADIO
# ============================================================

import os
import sys
import subprocess

# ------------------------------------------------------------
# 1. Import required libraries
# ------------------------------------------------------------

try:
    import tensorflow as tf
except ModuleNotFoundError:
    print("TensorFlow not found. Installing it...")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "tensorflow"]
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            "TensorFlow installation failed. Please install it manually with: "
            "pip install tensorflow"
        ) from e

    import tensorflow as tf
import numpy as np
import gradio as gr

from PIL import Image, ImageOps
from google import genai


# ------------------------------------------------------------
# 2. Gemini API Key
# ------------------------------------------------------------
# Make sure GEMINI_API_KEY was created in an earlier cell.
#
# Example:
# GEMINI_API_KEY = "YOUR_API_KEY_HERE"
#
# For safety, don't upload your actual API key to GitHub.

if "GEMINI_API_KEY" not in globals():
    if "GEMINI_API_KEY" not in globals():
        GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

        if not GEMINI_API_KEY:
            GEMINI_API_KEY = input("Enter your Gemini API key and press Enter: ").strip()

        if not GEMINI_API_KEY:
            raise ValueError(
                "GEMINI_API_KEY is not defined. "
                "Please create your Gemini API key variable first."
            )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ------------------------------------------------------------
# 3. CHECK MODEL
# ------------------------------------------------------------
# IMPORTANT:
# Your trained TensorFlow model must already be loaded.
#
# The variables expected here are:
#
#     infer
#     class_names
#
# If you already have a previous cell that creates these,
# run that cell before this code.
# ------------------------------------------------------------

if "infer" not in globals():
    if "infer" not in globals():
        model_candidates = [
            "keras_model.h5",
            "model.h5",
            "model.keras",
            "saved_model",
            os.path.join(os.getcwd(), "keras_model.h5"),
            os.path.join(os.getcwd(), "model.h5"),
            os.path.join(os.getcwd(), "model.keras"),
            os.path.join(os.getcwd(), "saved_model"),
        ]

        for candidate in model_candidates:
            if os.path.exists(candidate):
                try:
                    infer = tf.keras.models.load_model(candidate, compile=False)
                    print(f"Loaded TensorFlow model from: {candidate}")
                    break
                except Exception as exc:
                    print(f"Could not load model from {candidate}: {exc}")

        if "infer" not in globals():
            raise NameError(
                "The variable 'infer' was not found.\n\n"
                "Run the previous cell that loads your trained TensorFlow model before running this cell.\n"
                "Example: infer = tf.keras.models.load_model('keras_model.h5')"
            )

    if "class_names" not in globals():
        labels_candidates = [
            "labels.txt",
            os.path.join(os.getcwd(), "labels.txt"),
        ]

        for labels_path in labels_candidates:
            if os.path.exists(labels_path):
                with open(labels_path, "r", encoding="utf-8") as f:
                    class_names = [line.strip() for line in f if line.strip()]
                break

        if "class_names" not in globals():
            raise NameError(
                "The variable 'class_names' was not found.\n\n"
                "Run the previous cell that defines your disease class names before running this cell."
            )

if "class_names" not in globals():
    raise NameError(
        "The variable 'class_names' was not found.\n\n"
        "Run the previous cell that defines your disease "
        "class names before running this cell."
    )


# ------------------------------------------------------------
# 4. Prediction + Gemini explanation function
# ------------------------------------------------------------

def predict_disease_and_explain(image_file, language_input):

    # Check image
    if image_file is None:
        return None, "Please upload a plant leaf image."

    # Default language
    if not language_input:
        language_input = "English"


    # --------------------------------------------------------
    # Prepare image
    # --------------------------------------------------------

    data = np.ndarray(
        shape=(1, 224, 224, 3),
        dtype=np.float32
    )

    # Gradio may provide a filepath
    if isinstance(image_file, str):
        image_path = image_file
    else:
        image_path = image_file.name

    image = Image.open(image_path).convert("RGB")

    size = (224, 224)

    image_resized = ImageOps.fit(
        image,
        size,
        Image.Resampling.LANCZOS
    )

    image_array = np.asarray(image_resized)

    # Normalize exactly as expected by many Teachable Machine models
    normalized_image_array = (
        image_array.astype(np.float32) / 127.5
    ) - 1

    data[0] = normalized_image_array


    # --------------------------------------------------------
    # Run TensorFlow prediction
    # --------------------------------------------------------

    input_tensor = tf.convert_to_tensor(data)

    output = infer(input_tensor)

    # Handle dictionary output
    if isinstance(output, dict):
        prediction = list(output.values())[0].numpy()
    else:
        prediction = output.numpy()

    # Make sure prediction is 1D
    prediction = np.asarray(prediction).flatten()

    index = int(np.argmax(prediction))

    class_name = str(class_names[index]).strip()

    confidence_score = float(prediction[index])


    # --------------------------------------------------------
    # Determine healthy / unhealthy
    # --------------------------------------------------------

    if "Healthy" in class_name:

        plant_name = class_name.replace(
            "_Healthy", ""
        )

        status = "Healthy"

        prompt_for_llm = f"""
You are an expert agricultural plant pathologist
and crop-health advisor.

The AI image classifier identified the plant condition as:

{class_name}

Explain this result in simple language that a farmer
or gardener can understand.

Include:

- A brief explanation of the healthy condition
- What it means for the plant to be healthy
- General tips for maintaining plant health
- How to prevent common plant diseases

Plant: {plant_name}
Classification confidence: {confidence_score * 100:.2f}%

Answer ONLY in {language_input}.

Start your response with:

**Plant:** {plant_name}
**Status:** Healthy
**Confidence:** {confidence_score * 100:.2f}%
"""


    else:

        parts = class_name.split("_")

        if len(parts) > 1:
            plant_name = parts[0]
        else:
            plant_name = "Plant"

        disease_name = class_name

        status = "Unhealthy"

        prompt_for_llm = f"""
You are an expert agricultural plant pathologist
and crop-health advisor.

The AI image classifier identified the plant condition as:

{class_name}

Provide practical, safe, and easy-to-understand
information for a farmer or gardener.

Include:

- A brief explanation of the disease
- Key symptoms
- Immediate actions to take
- Natural or low-cost remedies
- When chemical treatment may be appropriate
- Preventive measures for future crops

Do not claim that the AI diagnosis is 100% certain.
Mention that the result should be confirmed by an
agricultural expert if the plant is valuable or the
symptoms are unclear.

Plant: {plant_name}
Disease: {disease_name}
Classification confidence: {confidence_score * 100:.2f}%

Answer ONLY in {language_input}.

Start your response with:

**Plant:** {plant_name}
**Status:** Unhealthy
**Disease:** {disease_name}
**Confidence:** {confidence_score * 100:.2f}%
"""


    # --------------------------------------------------------
    # Gemini API call
    # --------------------------------------------------------

    try:

        gemini_response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt_for_llm
        )

        explanation = gemini_response.text

    except Exception as e:

        explanation = (
            f"### Prediction\n\n"
            f"**Plant:** {plant_name}\n\n"
            f"**Status:** {status}\n\n"
            f"**Disease/Class:** {class_name}\n\n"
            f"**Confidence:** "
            f"{confidence_score * 100:.2f}%\n\n"
            f"### Gemini explanation unavailable\n\n"
            f"Error: `{str(e)}`"
        )


    # --------------------------------------------------------
    # Return results to Gradio
    # --------------------------------------------------------

    return image, explanation


# ------------------------------------------------------------
# 5. Create Gradio interface
# ------------------------------------------------------------

iface = gr.Interface(

    fn=predict_disease_and_explain,

    inputs=[

        gr.File(
            label="🌿 Upload a Plant Leaf Image",
            file_types=[".jpg", ".jpeg", ".png"]
        ),

        gr.Textbox(
            label="🌐 Enter Preferred Language",
            value="English",
            placeholder="Example: English, Tamil, Hindi..."
        )
    ],

    outputs=[

        gr.Image(
            label="📷 Uploaded Plant Image"
        ),

        gr.Markdown(
            label="🤖 AI-Powered Disease Explanation"
        )
    ],

    title="🌱 AI Plant Disease Detection",

    description=(
        "Upload a plant leaf image to detect possible plant "
        "diseases and receive an AI-generated explanation "
        "and practical guidance in your preferred language."
    ),

    flagging_mode="never"
)


# ------------------------------------------------------------
# 6. Launch application
# ------------------------------------------------------------

iface.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f25f8015e7b3491a28.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import tensorflow as tf
print(tf.__version__)

2.21.0


This code creates a simple Gradio interface:

*   `upload_image` is a function that takes the uploaded file object and returns its name.
*   `gr.File(label="Upload an Image")` creates an input component allowing users to upload a file.
*   `gr.Textbox(label="Uploaded File Path")` displays the path of the uploaded file.

To run this interface, uncomment `iface.launch()` in the code and execute the cell.

In [20]:
import gradio as gr

In [21]:
import os
print(os.getcwd())
print(os.listdir("."))

c:\Users\Sahaana\OneDrive\Desktop\AI-PLANT-DISEASE-DETECTION
['.git', '.gitignore', '.gradio', 'app.py', 'converted_savedmodel.zip', 'labels.txt', 'model.savedmodel', 'Plant_leave_diseases_dataset', 'Project 7-Plant Disease Detection-R.Sahaana.pdf', 'PROJECT_7_Notebook__AI_Powered_Plant_Disease_Detection (1).ipynb', 'README.md', 'requirements.txt', 'streamlit_app.py']


In [11]:
import zipfile, glob, os
zip_path = glob.glob("*.zip")[0]
zipfile.ZipFile(zip_path).extractall(".")
print(os.listdir("."))

['.git', '.gitignore', '.gradio', 'app.py', 'converted_savedmodel.zip', 'labels.txt', 'model.savedmodel', 'Plant_leave_diseases_dataset', 'Project 7-Plant Disease Detection-R.Sahaana.pdf', 'PROJECT_7_Notebook__AI_Powered_Plant_Disease_Detection (1).ipynb', 'README.md', 'requirements.txt', 'streamlit_app.py']


In [12]:
%pip install tensorflow gradio google-genai pillow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

<center>
$END$
</center>

---